# Knowledge distillation in Tensorflow

In [1]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import numpy as np

In [ ]:
# Teacher model (simpler model)
teacher_model = models.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

teacher_model.compile(optimizer=optimizers.Adam(),
                      loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# Assume the teacher model is already trained

In [3]:
# Student model (simpler model)
student_model = models.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(10, activation='softmax')
])

student_model.compile(optimizer=optimizers.Adam(),
                      loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Distillation loss function using TensorFlow operations
def distillation_loss(teacher_logits, student_logits, temperature=3):
    teacher_probs = tf.nn.softmax(teacher_logits / temperature)
    student_probs = tf.nn.softmax(student_logits / temperature)
    return tf.reduce_mean(tf.keras.losses.categorical_crossentropy(teacher_probs, student_probs))

In [5]:
# Train the student model using knowledge distillation with smaller data and fewer epochs
def train_student(student, teacher, x_train, y_train, batch_size=32, epochs=2, temperature=3):
    for epoch in range(epochs):
        num_batches = len(x_train) // batch_size
        for batch in range(num_batches):
            x_batch = x_train[batch * batch_size:(batch + 1) * batch_size]
            y_batch = y_train[batch * batch_size:(batch + 1) * batch_size]

            # Predict teacher logits for the batch
            teacher_logits = teacher.predict(x_batch)

            with tf.GradientTape() as tape:
                # Predict student logits for the batch
                student_logits = student(x_batch)
                # Calculate distillation loss
                loss = distillation_loss(teacher_logits, student_logits,
                                        temperature)

            # Apply gradients
            grads = tape.gradient(loss, student.trainable_variables)
            student.optimizer.apply_gradients(zip(grads,
                                                student.trainable_variables))

    print(f"Epoch {epoch + 1} completed. Loss: {loss.numpy()}")


In [6]:
# Use a smaller subset of the MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train[:1000], x_test[:1000] # Use only 1000 samples for quick training

x_train, x_test = x_train.astype('float32') / 255.0, x_test.astype('float32') / 255.0

# Train the student model
train_student(student_model, teacher_model, x_train, y_train,
              batch_size=32, epochs=2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━